In [7]:
#TODO make sure this renders in the github repo

# Data Loader

The decoder-only model (Llama 3 8B), is trained on next-token prediction using a causal mask: Given a continuous stream of tokens, we slice a fixed-length window (e.g., 256 tokens).

- **Input** ($X$): Tokens $0$ to $N-1$, e.g., $[x_0, x_1, ..., x_{N-1}]$
- **Target** ($Y$): Tokens $1$ to $N$, e.g., $[x_1, x_2, ..., x_{N}]$
- Timestamp example sequence: "This is the Llama 3 8B"
  - The **causal mask** ensures that the tokenized representation for token $i$ is calculated using only the tokens at positions $\leq i$, this prevents the model from "cheating" by looking at the target tokens during training.
  - **Note:** The timestamp table is only for visualization purposes, in practice, the model processes the entire window in one forward pass!

    | Timestamp | Input | Target |
    | --- | --- | --- |
    | 1 | "This" | "is" |
    | 2 | "This is" | "the" |
    | 3 | "This is the" | "Llama" |
    | 4 | "This is the Llama" | "3" |
    | 5 | "This is the Llama 3" | "8B" |

**Example of what the tokens will look like:**

```text
First 10 tokens of the Input:  
    [1, 269, 72, 224, 44, 81, 71, 72, 83, 262]
First 10 tokens of the Target (shifted left): 
    [269, 72, 224, 44, 81, 71, 72, 83, 262, 71]
```

- The dataloader yields dense, packed **batches of tokens**.

## Pipeline

1. **Pre-Tokenize Prior To Training:** Stream the dataset, tokenize the whole dataset, and pack it into fixed length chucks separated by `doc_end_token`.
2. **Binary Storage:** Save the tokenized outputs into a massive flat 1D array (of uint16, uint32, or higher) in a binary file.
- The `Dataloader` is simply a memory map to the binary file. With this method, the size of the dataset stored in memory is minimal. Now, most of memory is reserved for storing the model's weights and the massive gradient tensors generated during the backward pass.

⭐️ See [prepare_pretrained.py](../data_prep/prepare_pretrained.py) for pre-tokenize code.

⚠️⚠️ **WARNING:** If you plan on training on massive datasets such as the 15.6T tokens that Llama 3 pre-training used, you can not store that many tokens in a single binary file like I am for this project, instead use other methods like splitting the tokens into **shards** of binary files.

In [8]:
import easyjupyter
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [9]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from configs import ConfigTemplate

In [10]:
class TokenizedDatasetLoader(Dataset):
    def __init__(self, cfg: ConfigTemplate, bin_path, tokens_seen=0):
        """
        Retrieve data from a pre-tokenized dataset that is stored in a binary file.

        The data coming from the binary file is split into blocks of `seq_len`, it only loads what it needs, instead of loading the entire file into memory.
            For example:
                The binary is: [1, 456, 17569, 5364, 4, 2, 10, 12, 15, 18, 244, 1242, ...]
                And the seq_len = 3
                    The first block would be: [1, 456, 17569]
                    The second block would be: [5364, 4, 2]
                    The third block would be: [10, 12, 15]
                    etc...

        Args:
            bin_path: Path to the binary file containing the pre-tokenized dataset.
            tokens_seen: The number of tokens the model has already trained on for this specific binary file. This is used to skip those tokens.
                - Note, the tokens pulled from the binary file come in non-overlapping blocks of seq_len, so if `seq_len` = 4,096 and you set `tokens_seen` = 4, it will evaluate this 4 // 4,096 = 0 seq_len seen, meaning you still get the first seq_len block.
        """
        self.cfg = cfg
        # Memory-map the binary file
        self.data = np.memmap(bin_path, dtype=cfg.dtype, mode="r")

        # calculate how many samples (full context windows/sequences blocks) the model has seen.
        self.blocks_seen = tokens_seen // cfg.seq_len

        # Calculate the total possibles samples in the binary file.
        self.total_possible_blocks = (len(self.data) - 1) // cfg.seq_len

        # Calculate how many full sequence window blocks we have left in the binary file.
        self.num_samples = max(0, self.total_possible_blocks - self.blocks_seen)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        """Get a single context window from the dataset."""

        # Shift the index by the number of samples already seen, if needed.
        actual_idx = idx + self.blocks_seen

        # Ensure we don't go out of bounds.
        if actual_idx >= self.total_possible_blocks:
            raise IndexError("Index out of range")

        # Stride by the sequence length.
        start = actual_idx * self.cfg.seq_len
        end = start + self.cfg.seq_len + 1

        chunk = torch.tensor(self.data[start:end], dtype=torch.int64)

        # Create the casual mask
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

In [11]:
def create_pretrain_dataloader(cfg, bin_path, tokens_seen=0):
    """
    Args:
        bin_path: The path to the binary file that contains the dataset.
        tokens_seen: The number of tokens the model has already trained on for this specific binary file.
            - Note, the tokens pulled from the binary file come in non-overlapping blocks of seq_len, so if `seq_len` = 4,096 and you set `tokens_seen` = 4, it will evaluate this 4 // 4,096 = 0 seq_len seen, meaning you still get the first seq_len block.
    """

    dataset = TokenizedDatasetLoader(cfg, bin_path, tokens_seen=tokens_seen)

    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        # pin_memory=True is not supported on mps
        pin_memory=False if cfg.device.type == "mps" else True,
    )

## Test

`Run this to create a small binary dataset`

```bash
python prepare.py --stage pretrain --overfit
```

In [12]:
# @i-c
from configs import Llama3_1_405B  # TODO change to scaled down
from model.bpe_tokenizer import BPETokenizer

cfg = Llama3_1_405B.initialize(is_overfit=True)

# Lower seq_len to be able to see the blocks of data
cfg.seq_len = 8
cfg.batch_size = 1

# Get a trained tokenizer
t = BPETokenizer(cfg)
success, tokenizer = t.load_tokenizer()

if not (cfg.data_dir / "overfit_initial_stage.bin").exists():
    raise FileNotFoundError("Please run `python prepare.py --stage pretrain --overfit` first to create the small overfit dataset")
    
dataloader = create_pretrain_dataloader(cfg=cfg, bin_path=cfg.data_dir / "overfit_initial_stage.bin")

# Fetch a single batch
print("Fetching a single batch from FineWeb-Edu...")
data_iterator = iter(dataloader)
x, y = next(data_iterator)
print(f"batch_size={cfg.batch_size}, seq_len={cfg.seq_len}")


Project Root: /Users/tonyavis/Main/Build_an_LLM
Using config: Llama 3.1 405B
Configured for overfitting...

Loading existing trained BPE Tokenizer with vocab size: 128000 (Trained on 100 documents) from: (/Users/tonyavis/Main/Build_an_LLM/artifacts/universal_tokenizer/bpe_vocab_size_128000_samples_100.json)...
Fetching a single batch from FineWeb-Edu...
batch_size=1, seq_len=8


In [13]:
# @i-c
print("A single batch of data:")
print("x.shape:", x.shape, "y.shape:", y.shape)
print(f"\nx: {x}")
print(f"y: {y}")
print(f"\n The `y` is shifted by one token to the left")

A single batch of data:
x.shape: torch.Size([1, 8]) y.shape: torch.Size([1, 8])

x: tensor([[    1,   456, 17569,  2526,   202,  2776,   522,   266]])
y: tensor([[  456, 17569,  2526,   202,  2776,   522,   266,  4493]])

 The `y` is shifted by one token to the left


In [14]:
# @i-c:
print(f"Decoding the first {cfg.seq_len} tokens of x[0] to view the document:")
token_ids_list = x[0][:cfg.seq_len].tolist()
decoded_document = tokenizer.decode(token_ids_list)
print(f"\nDecoded document: \n\n[{decoded_document}...]")

Decoding the first 8 tokens of x[0] to view the document:

Decoded document: 

[The Independent Jane
For all the...]


### Visualize blocks with the same `tokens_seen`

In [16]:
# @i-c
def visualize_blocks_with_same_tokens_seen(cfg, n_blocks=3):
    dataloader = create_pretrain_dataloader(cfg, bin_path=cfg.data_dir / "overfit_initial_stage.bin", tokens_seen=0)
    data_iter = iter(dataloader)

    print(f"--- tokens_seen = {0} | seq_len = {cfg.seq_len} ---")
    print(f"This is what the data would look like if you are training a new model from the very start.")
    print(f"They y is the same as the x, except the y is one token shifted to the left")
    try:
        for i in range(n_blocks):
            x, y = next(data_iter)
            print(f"  Block {i + 1}:")
            print(f"    x: {x[0].tolist()}")
            print(f"    y: {y[0].tolist()}")
    except StopIteration:
        print("End of the dataset binary file reached")
        
    
visualize_blocks_with_same_tokens_seen(cfg, n_blocks=5)

--- tokens_seen = 0 | seq_len = 8 ---
This is what the data would look like if you are training a new model from the very start.
They y is the same as the x, except the y is one token shifted to the left
  Block 1:
    x: [1, 456, 17569, 2526, 202, 2776, 522, 266]
    y: [456, 17569, 2526, 202, 2776, 522, 266, 4493]
  Block 2:
    x: [4493, 15, 17937, 294, 13871, 289, 2526, 3812]
    y: [15, 17937, 294, 13871, 289, 2526, 3812, 421]
  Block 3:
    x: [421, 86, 3947, 15, 886, 573, 364, 2644]
    y: [86, 3947, 15, 886, 573, 364, 2644, 607]
  Block 4:
    x: [607, 311, 4321, 294, 2302, 17, 17570, 281]
    y: [311, 4321, 294, 2302, 17, 17570, 281, 2572]
  Block 5:
    x: [2572, 294, 266, 4321, 291, 4834, 17, 202]
    y: [294, 266, 4321, 291, 4834, 17, 202, 17754]


### Test `seq_len` blocks with different `tokens_seen`

`tokens_seen` is used to skip blocks the model has already been trained on.

In [17]:
# @i-c
tokens_seen = 6
print(f"seq_len = {cfg.seq_len} | tokens_seen = {tokens_seen}")

seq_len = 8 | tokens_seen = 6


In [19]:
# @i-c
def visualize_blocks_with_different_tokens_seen(cfg, tokens_seen_list, n_blocks=2):
    """
    Creates a dataloader for various `tokens_seen` values and prints the resulting retrieved x and y blocks
    """
    print(f"This is what the data would look like if you continue to train from a checkpoint (i.e., model was perviously trained).")
    print(f"They y is the same as the x, except the y is one token shifted to the left")
    for i in tokens_seen_list:
        dataloader = create_pretrain_dataloader(cfg, bin_path=cfg.data_dir / "overfit_initial_stage.bin", tokens_seen=i)
        data_iter = iter(dataloader)

        blocks_skipped = i // cfg.seq_len
        print(f"\n--- tokens_seen={i} | Blocks Skipped={blocks_skipped} | seq_len = {cfg.seq_len} ---")
        if i != 0:
            print(f"Continue training. The model had perviously been trained on the first {i} tokens, it will continue training with these blocks, that it has not seen yet")
        else:
            print("Training a new model from the very start of data")
        try:
            for i in range(n_blocks):
                x, y = next(data_iter)
                print(f"  Block {i + 1 + blocks_skipped}:")
                print(f"    x: {x[0].tolist()}")
                print(f"    y: {y[0].tolist()}")
        except StopIteration:
            print("End of the dataset binary file reached")
            
    
test_tokens_seen = [0, 4, 8, 10, 16, 24]
visualize_blocks_with_different_tokens_seen(cfg, test_tokens_seen, n_blocks=4)

This is what the data would look like if you continue to train from a checkpoint (i.e., model was perviously trained).
They y is the same as the x, except the y is one token shifted to the left

--- tokens_seen=0 | Blocks Skipped=0 | seq_len = 8 ---
Training a new model from the very start of data
  Block 1:
    x: [1, 456, 17569, 2526, 202, 2776, 522, 266]
    y: [456, 17569, 2526, 202, 2776, 522, 266, 4493]
  Block 2:
    x: [4493, 15, 17937, 294, 13871, 289, 2526, 3812]
    y: [15, 17937, 294, 13871, 289, 2526, 3812, 421]
  Block 3:
    x: [421, 86, 3947, 15, 886, 573, 364, 2644]
    y: [86, 3947, 15, 886, 573, 364, 2644, 607]
  Block 4:
    x: [607, 311, 4321, 294, 2302, 17, 17570, 281]
    y: [311, 4321, 294, 2302, 17, 17570, 281, 2572]

--- tokens_seen=4 | Blocks Skipped=0 | seq_len = 8 ---
Continue training. The model had perviously been trained on the first 4 tokens, it will continue training with these blocks, that it has not seen yet
  Block 1:
    x: [1, 456, 17569, 2526, 20

In [21]:
# @i-c
data = np.memmap(cfg.data_dir / "overfit_initial_stage.bin", dtype=cfg.dtype, mode="r")
tokens_seen = 16
data

memmap([    1,   456, 17569, ...,  4281,   281,  1019],
       shape=(10000,), dtype=uint32)

In [22]:
# @i-c
blocks_seen = tokens_seen // cfg.seq_len
print(blocks_seen)

2


In [23]:
# @i-c
# Calculate the total possibles samples in the binary file.
total_possible_blocks = (len(data) - 1) // cfg.seq_len
total_possible_blocks

1249

In [24]:
# @i-c
# Calculate how many full context windows blocks we have left in the binary file.
num_samples = max(0, total_possible_blocks - blocks_seen)
num_samples

1247